# HTB network-log transformation and 60/20/20 split

Module 292, Section 7. Use the **AI** kernel. This is an annotated companion to HTB's one-hot, `log1p`, and two-stage split examples. It loads only HTB's small course CSV, drops rows that fail the prior section's domain checks, and never trains a model. The original CSV and notebook outputs are not saved to the repo.

## Load and validate the course sample

Supply `HTB_DEMO_DATASET_ZIP` to use an existing local download; otherwise this cell fetches the exact course ZIP in memory. We bound the archive member size and require the expected columns. This is input preparation, not the new transformation itself.

In [ ]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import os
import re
import numpy as np
import pandas as pd
import requests

course_url = 'https://cdn.services-k8s.prod.aws.htb.systems/content/modules/292/demo_dataset.zip'
local_zip = Path(os.environ.get('HTB_DEMO_DATASET_ZIP', 'demo_dataset.zip'))
if local_zip.is_file():
    archive_bytes = local_zip.read_bytes()
else:
    response = requests.get(course_url, timeout=30)
    response.raise_for_status()
    archive_bytes = response.content
with ZipFile(BytesIO(archive_bytes)) as archive:
    member = archive.getinfo('demo_dataset.csv')
    if member.file_size > 10_000_000:
        raise ValueError('Unexpectedly large course CSV')
    with archive.open(member) as source:
        raw = pd.read_csv(source)
required = {'log_id', 'source_ip', 'destination_port', 'protocol', 'bytes_transferred', 'threat_level'}
assert required.issubset(raw.columns)
print('Raw rows:', len(raw))

## Carry forward the drop-invalid branch

The prior lesson found 23 distinct invalid rows in the 100-row sample. A value can fail more than one test, so combine Boolean conditions *per row*. `log_id` is an identifier, not a model feature; `source_ip` is checked here but later excluded from the demonstration feature table to avoid host memorization. Unknown `threat_level` labels are dropped, never guessed.

In [ ]:
ipv4 = re.compile(r'^((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)$')
allowed_protocols = {'TCP', 'TLS', 'SSH', 'POP3', 'DNS', 'HTTPS', 'SMTP', 'FTP', 'UDP', 'HTTP'}
ports = pd.to_numeric(raw['destination_port'], errors='coerce')
byte_counts = pd.to_numeric(raw['bytes_transferred'], errors='coerce')
labels = pd.to_numeric(raw['threat_level'], errors='coerce')
valid = (raw['source_ip'].map(lambda v: pd.notna(v) and bool(ipv4.fullmatch(str(v))))
         & ports.notna() & ports.between(0, 65535) & (ports % 1 == 0)
         & raw['protocol'].isin(allowed_protocols)
         & byte_counts.notna() & (byte_counts >= 0) & (byte_counts % 1 == 0)
         & labels.isin([0, 1, 2]))
clean = raw.loc[valid].copy().reset_index(drop=True)
clean['destination_port'] = ports.loc[valid].to_numpy(dtype=int)
clean['bytes_transferred'] = byte_counts.loc[valid].to_numpy(dtype=int)
clean['threat_level'] = labels.loc[valid].to_numpy(dtype=int)
print('Valid, originally labeled rows:', len(clean))
if len(raw) == 100:
    assert len(clean) == 77  # The course sample's Section 6 result.
assert clean['threat_level'].isin([0, 1, 2]).all()

## Separate features and answers, then split before learning encodings

`X` (say **capital X**) holds predictors; `y` (say **lowercase y**) holds the answer labels. `X_train` means **X sub train**. The first split reserves 20% for final test; the second reserves 25% of the remaining 80% for validation. Read `0.8 × 0.25 = 0.2` as **eight-tenths times one-quarter equals two-tenths**, so the intended full-data ratio is 60/20/20. Rounding makes 77 rows land near, not exactly on, those percentages. HTB encodes before splitting; here we split first to avoid learning category mappings from held-out rows.

In [ ]:
from sklearn.model_selection import train_test_split

# Identifiers may become shortcuts, so this teaching feature table excludes them.
X = clean[['destination_port', 'protocol', 'bytes_transferred']].copy()
y = clean['threat_level'].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1337)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=1337)
print('Train / validation / test:', len(X_train), len(X_val), len(X_test))
assert len(X_train) + len(X_val) + len(X_test) == len(clean)
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)

## One-hot encode `protocol` with training-only categories

One-hot means one binary indicator per known category: for two learned categories TCP and UDP, TCP → `[1, 0]`, UDP → `[0, 1]`. A category code of 2 would otherwise look numerically larger than a category code of 1. `fit` learns category names from **training rows only**; `transform` applies the same columns to all three folds. `handle_unknown='ignore'` makes a truly unseen category all-zero in those learned columns, rather than raising an error. `reset_index(drop=True)` prevents accidental row misalignment when joining encoded columns back to numeric columns.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[['protocol']])  # Learn categories on training rows only.
feature_names = encoder.get_feature_names_out(['protocol'])

def encode_fold(fold):
    indicator = pd.DataFrame(encoder.transform(fold[['protocol']]),
                             columns=feature_names, index=fold.index)
    # Index-preserving concat keeps each event paired with its own indicators.
    return pd.concat([fold.drop(columns='protocol'), indicator], axis=1)

train_encoded = encode_fold(X_train)
val_encoded = encode_fold(X_val)
test_encoded = encode_fold(X_test)
print('Learned protocol indicators:', list(feature_names))
assert train_encoded.columns.equals(val_encoded.columns)
assert train_encoded.columns.equals(test_encoded.columns)
assert (train_encoded[feature_names].sum(axis=1) == 1).all()
print('Unseen validation protocols:', int((val_encoded[feature_names].sum(axis=1) == 0).sum()))

## Compress the skewed byte count

Formula: `z = ln(1 + b)`. Say **z equals the natural logarithm of one plus b**. `b` is a nonnegative byte count; `1 + b` permits zero; `z` is the compressed result. Thus 0 → 0, 9 → about 2.30, and 999 → about 6.91. `np.log1p` implements this formula. It is deterministic—there is no statistic to learn from the test fold—and `np.expm1` approximately reverses it. We transform the three folds separately and verify finite, nonnegative values.

In [ ]:
def log_bytes(encoded_fold):
    transformed = encoded_fold.copy()
    original = transformed['bytes_transferred'].to_numpy(dtype=float)
    assert np.isfinite(original).all() and (original >= 0).all()
    transformed['bytes_transferred'] = np.log1p(original)  # z = ln(1 + b)
    assert np.allclose(np.expm1(transformed['bytes_transferred']), original)
    return transformed

X_train_ready = log_bytes(train_encoded)
X_val_ready = log_bytes(val_encoded)
X_test_ready = log_bytes(test_encoded)
print('Original byte-count range:', X['bytes_transferred'].min(), X['bytes_transferred'].max())
print('Log-transformed training range:',
      X_train_ready['bytes_transferred'].min(), X_train_ready['bytes_transferred'].max())
assert np.isfinite(X_train_ready.to_numpy(dtype=float)).all()
assert X_train_ready.shape[1] == X_val_ready.shape[1] == X_test_ready.shape[1]

The three `X_*_ready` tables and matching `y_*` label Series illustrate the section's transformations and split; no model is fitted. The final test fold should remain untouched during model selection. A random row split is the HTB teaching choice, but a real intrusion detector may require grouping events by host/incident or splitting by time to avoid near-duplicate leakage. A newly seen protocol or a shifted byte-count distribution should be monitored rather than silently treated as proof of good generalization.